In [32]:
import pandas as pd
import csv
import numpy as np
import GPy
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

## Initial Data Import

In [ ]:
pd.set_option('display.max_rows', 500)
data = pd.read_csv('../MD_csvs/polymer_chi_database.csv')
data

In [ ]:
gtt_data = pd.read_csv('../MD_csvs/glass_transition_temp.csv')
gtt_data

In [8]:
data.isna().sum()

Polymer             0
Nickname          321
Si                  0
N                   0
chi0                0
chi0v               0
chi1                0
chi1v               0
chi0v_Si            0
chi1v_Si            0
density           209
molar_volume      209
N_MV              209
delta_avg         313
E_coh_fedors      218
E_coh_krevelen    218
dtype: int64

## Krevelen Cohesive Energy

In [ ]:
X_krev = (
    data.dropna(subset=['E_coh_krevelen'])
        [['chi0', 'chi1', 'chi0v_Si', 'chi1v_Si']]
)
X_krev

In [ ]:
y_krev = (
    data.dropna(subset=['E_coh_krevelen'])
        [['E_coh_krevelen']]
)
y_krev

In [11]:
print(min(data.dropna(subset=['E_coh_krevelen'])['E_coh_krevelen']))
print(max(data.dropna(subset=['E_coh_krevelen'])['E_coh_krevelen']))

6720.0
263600.0


In [12]:
def make_plot(y_test, y_pred):
    plt.style.use('fivethirtyeight')
    x = np.arange(5000,270000,1)
    y = x
    plt.plot(x,y,'k--')
    plt.scatter(y_test, y_pred, c='r')
    plt.ylabel('Predicted Krevelen Cohesive Energy')
    plt.xlabel('Actual Krevelen Cohesive Energy')
    plt.show()

In [19]:
# Gaussian Process models with multiple kernels

X_train, X_test, y_train, y_test = train_test_split(
    X_krev, y_krev, test_size=0.2, random_state=53
)

kernels = {
    "Exponential": GPy.kern.Exponential(input_dim=X_train.shape[1], ARD=True),
    "Matern32": GPy.kern.Matern32(input_dim=X_train.shape[1], ARD=True),
    "Matern52": GPy.kern.Matern52(input_dim=X_train.shape[1], ARD=True),
    "RBF": GPy.kern.RBF(input_dim=X_train.shape[1], ARD=True),
}

results = {}

for kname, kern in kernels.items():
    model = GPy.models.GPRegression(X_train.values, y_train.values.reshape(-1,1), kern)
    model.optimize_restarts(num_restarts=5, verbose=True)

    y_pred, _ = model.predict(X_test.values)
    X_pred, _ = model.predict(X_train.values)
    
    y_pred = y_pred.flatten()
    X_pred = X_pred.flatten()

    q2 = r2_score(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_train, X_pred)

    results[kname] = (q2, rmse, r2)

for k, v in results.items():
    print(k, "Q2:", v[0], "RMSE:", v[1], "R2:", v[2])


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:166: RuntimeWarning:overflow encountered in divide
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add


Optimization restart 1/5, f = 152549.04988164798
Optimization restart 2/5, f = 7630.822309738417
Optimization restart 3/5, f = 50526.55108327515
Optimization restart 4/5, f = 4123.144748943351
Optimization restart 5/5, f = 4687423.277879219
Optimization restart 1/5, f = 1296.7230582327707
Optimization restart 2/5, f = 6978.7411601929125


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:166: RuntimeWarning:overflow encountered in divide
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add


Optimization restart 3/5, f = 1298.964436547305
Optimization restart 4/5, f = 1290.784800694729
Optimization restart 5/5, f = 3460567.9163894993


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:166: RuntimeWarning:overflow encountered in divide


Optimization restart 1/5, f = 8069754.510444114
Optimization restart 2/5, f = 1743003.0713964545
Optimization restart 3/5, f = 2586.841539829884
Optimization restart 4/5, f = 141736.05453993136
Optimization restart 5/5, f = 6558369.20890732
Optimization restart 1/5, f = 1779010.3660273633


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:166: RuntimeWarning:overflow encountered in divide
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add


Optimization restart 2/5, f = 1295.006678908287
Optimization restart 3/5, f = 1289.939102837151
Optimization restart 4/5, f = 1293.5580684040135
Optimization restart 5/5, f = 32187.556538613117
Exponential Q2: 0.7047998933548183 RMSE: 25741.746621075003 R2: 0.8422650614689334
Matern32 Q2: 0.7286712393067704 RMSE: 24679.006464536124 R2: 0.8284780647748109
Matern52 Q2: 0.7152676688804718 RMSE: 25281.226990495277 R2: 0.8155561810771895
RBF Q2: 0.714543249130884 RMSE: 25313.3669761947 R2: 0.835864461181421


In [25]:
# Gaussian Process models with multiple kernels

X_train, X_test, y_train, y_test = train_test_split(
    X_krev, y_krev, test_size=0.2, random_state=53
)

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
# y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

kernels = {
    "Exponential": GPy.kern.Exponential(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern32": GPy.kern.Matern32(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern52": GPy.kern.Matern52(input_dim=X_train_scaled.shape[1], ARD=True),
    "RBF": GPy.kern.RBF(input_dim=X_train_scaled.shape[1], ARD=True),
}

results = {}

for kname, kern in kernels.items():
    model = GPy.models.GPRegression(X_train_scaled, y_train_scaled, kern)
    model.optimize_restarts(num_restarts=5, verbose=True)

    y_test_pred_scaled, _ = model.predict(X_test_scaled)
    # Convert back to original scale
    y_test_pred = y_scaler.inverse_transform(y_test_pred_scaled).flatten()
    
    y_train_pred_scaled, _ = model.predict(X_train_scaled)
    # Convert back to original scale
    y_train_pred = y_scaler.inverse_transform(y_train_pred_scaled).flatten()

    q2 = r2_score(y_test, y_test_pred)
    rmse = mean_squared_error(y_test, y_test_pred, squared=False)
    r2 = r2_score(y_train, y_train_pred)

    results[kname] = (q2, rmse, r2)

for k, v in results.items():
    print(k, "Q2:", v[0], "RMSE:", v[1], "R2:", v[2])


Optimization restart 1/5, f = 45.04907341027927
Optimization restart 2/5, f = 46.979981873616985
Optimization restart 3/5, f = 46.97998955327804
Optimization restart 4/5, f = 46.97998518351239
Optimization restart 5/5, f = 46.97998246998645
Optimization restart 1/5, f = 48.81404234455136
Optimization restart 2/5, f = 48.81404320354395
Optimization restart 3/5, f = 48.8140426624766
Optimization restart 4/5, f = 48.81404232017838
Optimization restart 5/5, f = 49.44508552813965
Optimization restart 1/5, f = 50.14328542366821
Optimization restart 2/5, f = 50.143285070638505
Optimization restart 3/5, f = 50.143285096496285
Optimization restart 4/5, f = 50.14328560631351
Optimization restart 5/5, f = 50.143286479719954
Optimization restart 1/5, f = 53.894284441461885
Optimization restart 2/5, f = 48.12263766852175
Optimization restart 3/5, f = 53.894284298984914
Optimization restart 4/5, f = 53.89428467033941
Optimization restart 5/5, f = 53.894284291825706
Exponential Q2: 0.7230378759297305

In [38]:
# Gaussian Process models with multiple kernels

X_train, X_test, y_train, y_test = train_test_split(
    X_krev, y_krev, test_size=0.2, random_state=53
)

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
# y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

kernels = {
    "Exponential": GPy.kern.Exponential(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern32": GPy.kern.Matern32(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern52": GPy.kern.Matern52(input_dim=X_train_scaled.shape[1], ARD=True),
    "RBF": GPy.kern.RBF(input_dim=X_train_scaled.shape[1], ARD=True),
}

results = {}

for kname, kern in kernels.items():
    model = GPy.models.GPRegression(X_train_scaled, y_train_scaled, kern)

    start_time = time.perf_counter()
    model.optimize_restarts(num_restarts=5, verbose=True)
    end_time = time.perf_counter()

    train_time = end_time - start_time

    y_test_pred_scaled, _ = model.predict(X_test_scaled)
    y_test_pred = y_scaler.inverse_transform(y_test_pred_scaled).flatten()
    
    y_train_pred_scaled, _ = model.predict(X_train_scaled)
    y_train_pred = y_scaler.inverse_transform(y_train_pred_scaled).flatten()

    q2 = r2_score(y_test, y_test_pred)
    rmse = mean_squared_error(y_test, y_test_pred, squared=False)
    r2 = r2_score(y_train, y_train_pred)

    # Extract hyperparameters
    lengthscales = model.kern.lengthscale.values
    variance = model.kern.variance.values[0]
    noise = model.likelihood.variance.values[0]

    results[kname] = {
        "Q2": q2,
        "RMSE": rmse,
        "R2": r2,
        "train_time_sec": train_time,
        "lengthscales": lengthscales,
        "variance": variance,
        "noise": noise
    }
    
for k, v in results.items():
    print(f"\nKernel: {k}")
    print("Train time (sec):", round(v["train_time_sec"], 3))
    print("Q2:", v["Q2"])
    print("RMSE:", v["RMSE"])
    print("R2:", v["R2"])
    print("Lengthscales:", v["lengthscales"])
    print("Kernel variance:", v["variance"])
    print("Noise variance:", v["noise"])

Optimization restart 1/5, f = 45.049073942537596
Optimization restart 2/5, f = 46.979982391891085
Optimization restart 3/5, f = 45.083167266373486
Optimization restart 4/5, f = 46.97998389043873
Optimization restart 5/5, f = 46.979982082155104
Optimization restart 1/5, f = 48.8140423442769
Optimization restart 2/5, f = 48.814042624761726
Optimization restart 3/5, f = 48.814043562536696
Optimization restart 4/5, f = 48.81404400865948
Optimization restart 5/5, f = 48.81404250799759
Optimization restart 1/5, f = 50.143285421452696
Optimization restart 2/5, f = 50.14328517558024
Optimization restart 3/5, f = 48.41656325188382
Optimization restart 4/5, f = 50.14328749354774
Optimization restart 5/5, f = 50.14328756887503
Optimization restart 1/5, f = 53.8942844414843
Optimization restart 2/5, f = 53.89429919144602
Optimization restart 3/5, f = 53.894296351438456
Optimization restart 4/5, f = 53.89428507799935
Optimization restart 5/5, f = 53.89428428470869

Kernel: Exponential
Train time (s

## Fedors Cohesive Energy

In [28]:
X_fed = (
    data.dropna(subset=['E_coh_fedors'])
        [['chi0', 'chi1', 'chi0v_Si', 'chi1v_Si']]
)

y_fed = (
    data.dropna(subset=['E_coh_fedors'])
        [['E_coh_fedors']]
)

print(min(data.dropna(subset=['E_coh_fedors'])['E_coh_fedors']))
print(max(data.dropna(subset=['E_coh_fedors'])['E_coh_fedors']))

8290.0
242995.0


In [35]:
# Gaussian Process models with multiple kernels

X_train, X_test, y_train, y_test = train_test_split(
    X_fed, y_fed, test_size=0.2, random_state=53
)

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
# y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

kernels = {
    "Exponential": GPy.kern.Exponential(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern32": GPy.kern.Matern32(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern52": GPy.kern.Matern52(input_dim=X_train_scaled.shape[1], ARD=True),
    "RBF": GPy.kern.RBF(input_dim=X_train_scaled.shape[1], ARD=True),
}

results = {}

for kname, kern in kernels.items():
    model = GPy.models.GPRegression(X_train_scaled, y_train_scaled, kern)

    start_time = time.perf_counter()
    model.optimize_restarts(num_restarts=5, verbose=True)
    end_time = time.perf_counter()

    train_time = end_time - start_time

    y_test_pred_scaled, _ = model.predict(X_test_scaled)
    y_test_pred = y_scaler.inverse_transform(y_test_pred_scaled).flatten()
    
    y_train_pred_scaled, _ = model.predict(X_train_scaled)
    y_train_pred = y_scaler.inverse_transform(y_train_pred_scaled).flatten()

    q2 = r2_score(y_test, y_test_pred)
    rmse = mean_squared_error(y_test, y_test_pred, squared=False)
    r2 = r2_score(y_train, y_train_pred)

    # Extract hyperparameters
    lengthscales = model.kern.lengthscale.values
    variance = model.kern.variance.values[0]
    noise = model.likelihood.variance.values[0]

    results[kname] = {
        "Q2": q2,
        "RMSE": rmse,
        "R2": r2,
        "train_time_sec": train_time,
        "lengthscales": lengthscales,
        "variance": variance,
        "noise": noise
    }
    
for k, v in results.items():
    print(f"\nKernel: {k}")
    print("Train time (sec):", round(v["train_time_sec"], 3))
    print("Q2:", v["Q2"])
    print("RMSE:", v["RMSE"])
    print("R2:", v["R2"])
    print("Lengthscales:", v["lengthscales"])
    print("Kernel variance:", v["variance"])
    print("Noise variance:", v["noise"])

Optimization restart 1/5, f = 31.00568186338267
Optimization restart 2/5, f = 31.615970539267906


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:166: RuntimeWarning:overflow encountered in divide
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add


Optimization restart 3/5, f = 31.00568319353681
Optimization restart 4/5, f = 31.00568203549045
Optimization restart 5/5, f = 31.005681994899184
Optimization restart 1/5, f = 26.313677010996038
Optimization restart 2/5, f = 26.425056489832087


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:243: RuntimeWarning:invalid value encountered in divide


Optimization restart 3/5, f = 26.313685715614398
Optimization restart 4/5, f = 26.31367775663572
Optimization restart 5/5, f = 26.31369225764152
Optimization restart 1/5, f = 25.765181478865408
Optimization restart 2/5, f = 25.76518074079349
Optimization restart 3/5, f = 25.765185527778392
Optimization restart 4/5, f = 25.778436176206746


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:586: RuntimeWarning:invalid value encountered in multiply


Optimization restart 5/5, f = 25.812598135692618
Optimization restart 1/5, f = 25.34563768802792
Optimization restart 2/5, f = 25.418545911240024
Optimization restart 3/5, f = 25.34563318945864
Optimization restart 4/5, f = 25.418541831186765
Optimization restart 5/5, f = 25.34563339944537

Kernel: Exponential
Train time (sec): 0.763
Q2: 0.8743008921788709
RMSE: 16200.416585943587
R2: 0.9999999999999958
Lengthscales: [1.14841407e+01 2.77203666e+01 2.39606383e+05 7.05437391e+00]
Kernel variance: 6.028837274893318
Noise variance: 5.562684646268137e-309

Kernel: Matern32
Train time (sec): 1.274
Q2: 0.8861630184712308
RMSE: 15417.067382716665
R2: 0.9290332936761756
Lengthscales: [2.30045418e+04 2.21246744e+01 2.92775102e+04 2.11526388e+01]
Kernel variance: 26.584225579635557
Noise variance: 0.07515200314810014

Kernel: Matern52
Train time (sec): 1.051
Q2: 0.8869306022589747
RMSE: 15365.002109875353
R2: 0.9269459518448708
Lengthscales: [3.67952088e+01 1.42316554e+01 2.72401907e+04 1.2942331

In [33]:
end_time - start_time

array([43.39187789, 44.48348999, 45.70242476, 42.62004614, 45.2279098 ])

## Molar Volume

In [30]:
X_mv = (
    data.dropna(subset=['molar_volume'])
        [['chi0', 'chi1', 'chi0v_Si', 'chi1v_Si']]
)

y_mv = (
    data.dropna(subset=['molar_volume'])
        [['molar_volume']]
)

print(min(data.dropna(subset=['molar_volume'])['molar_volume']))
print(max(data.dropna(subset=['molar_volume'])['molar_volume']))

24.0
466.6


In [31]:
len(kernels)

4

In [34]:
# Gaussian Process models with multiple kernels

X_train, X_test, y_train, y_test = train_test_split(
    X_mv, y_mv, test_size=0.2, random_state=53
)

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
# y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

kernels = {
    "Exponential": GPy.kern.Exponential(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern32": GPy.kern.Matern32(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern52": GPy.kern.Matern52(input_dim=X_train_scaled.shape[1], ARD=True),
    "RBF": GPy.kern.RBF(input_dim=X_train_scaled.shape[1], ARD=True),
}

results = {}

for kname, kern in kernels.items():
    model = GPy.models.GPRegression(X_train_scaled, y_train_scaled, kern)

    start_time = time.perf_counter()
    model.optimize_restarts(num_restarts=5, verbose=True)
    end_time = time.perf_counter()

    train_time = end_time - start_time

    y_test_pred_scaled, _ = model.predict(X_test_scaled)
    y_test_pred = y_scaler.inverse_transform(y_test_pred_scaled).flatten()
    
    y_train_pred_scaled, _ = model.predict(X_train_scaled)
    y_train_pred = y_scaler.inverse_transform(y_train_pred_scaled).flatten()

    q2 = r2_score(y_test, y_test_pred)
    rmse = mean_squared_error(y_test, y_test_pred, squared=False)
    r2 = r2_score(y_train, y_train_pred)

    # Extract hyperparameters
    lengthscales = model.kern.lengthscale.values
    variance = model.kern.variance.values[0]
    noise = model.likelihood.variance.values[0]

    results[kname] = {
        "Q2": q2,
        "RMSE": rmse,
        "R2": r2,
        "train_time_sec": train_time,
        "lengthscales": lengthscales,
        "variance": variance,
        "noise": noise
    }
    
for k, v in results.items():
    print(f"\nKernel: {k}")
    print("Train time (sec):", round(v["train_time_sec"], 3))
    print("Q2:", v["Q2"])
    print("RMSE:", v["RMSE"])
    print("R2:", v["R2"])
    print("Lengthscales:", v["lengthscales"])
    print("Kernel variance:", v["variance"])
    print("Noise variance:", v["noise"])

Optimization restart 1/5, f = 12.868071825787815
Optimization restart 2/5, f = 12.868071823920175
Optimization restart 3/5, f = 12.86807182317309
Optimization restart 4/5, f = 12.868071823495562
Optimization restart 5/5, f = 12.86807182433904
Optimization restart 1/5, f = 17.012710979845494
Optimization restart 2/5, f = 17.01271097982078
Optimization restart 3/5, f = 17.012710979740113
Optimization restart 4/5, f = 17.012710980885032
Optimization restart 5/5, f = 17.012710980782735
Optimization restart 1/5, f = 18.56063270198601
Optimization restart 2/5, f = 13.352179638874006
Optimization restart 3/5, f = 18.560632703795264


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:166: RuntimeWarning:overflow encountered in divide
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:586: RuntimeWarning:invalid value encountered in multiply


Optimization restart 4/5, f = 18.560632704993523
Optimization restart 5/5, f = 18.560632703621387
Optimization restart 1/5, f = 12.543496859247263
Optimization restart 2/5, f = 20.053171141390017
Optimization restart 3/5, f = 20.053171147489152
Optimization restart 4/5, f = 12.543497211667251
Optimization restart 5/5, f = 12.543496630133873

Kernel: Exponential
Train time (sec): 0.341
Q2: 0.9418198435441654
RMSE: 16.881664577956315
R2: 0.9999999999999957
Lengthscales: [13.31137442 28.5784511  47.46992689 47.90326889]
Kernel variance: 8.312553232797615
Noise variance: 3.295713124313795e-29

Kernel: Matern32
Train time (sec): 0.364
Q2: 0.9151784072698058
RMSE: 20.383606940276117
R2: 0.9891692644493639
Lengthscales: [0.95569901 1.59675353 4.96978269 8.16380017]
Kernel variance: 4.7739648304116855
Noise variance: 0.01916891731916266

Kernel: Matern52
Train time (sec): 0.388
Q2: 0.9634917626894444
RMSE: 13.372827946379537
R2: 0.9415544303552802
Lengthscales: [95.13421518 98.85989858 83.9932

## Glass Transition Temperature

In [36]:
X_gtt = gtt_data.drop(columns = ['Tg(exp)(K)', 'Tg(fit)(K)'])

y_gtt = gtt_data['Tg(exp)(K)']

print(min(gtt_data.dropna(subset=['Tg(exp)(K)'])['Tg(exp)(K)']))
print(max(gtt_data.dropna(subset=['Tg(exp)(K)'])['Tg(exp)(K)']))

130
685


In [37]:
# Gaussian Process models with multiple kernels

X_train, X_test, y_train, y_test = train_test_split(
    X_gtt, y_gtt, test_size=0.2, random_state=53
)

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
# y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

kernels = {
    "Exponential": GPy.kern.Exponential(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern32": GPy.kern.Matern32(input_dim=X_train_scaled.shape[1], ARD=True),
    "Matern52": GPy.kern.Matern52(input_dim=X_train_scaled.shape[1], ARD=True),
    "RBF": GPy.kern.RBF(input_dim=X_train_scaled.shape[1], ARD=True),
}

results = {}

for kname, kern in kernels.items():
    model = GPy.models.GPRegression(X_train_scaled, y_train_scaled, kern)

    start_time = time.perf_counter()
    model.optimize_restarts(num_restarts=5, verbose=True)
    end_time = time.perf_counter()

    train_time = end_time - start_time

    y_test_pred_scaled, _ = model.predict(X_test_scaled)
    y_test_pred = y_scaler.inverse_transform(y_test_pred_scaled).flatten()
    
    y_train_pred_scaled, _ = model.predict(X_train_scaled)
    y_train_pred = y_scaler.inverse_transform(y_train_pred_scaled).flatten()

    q2 = r2_score(y_test, y_test_pred)
    rmse = mean_squared_error(y_test, y_test_pred, squared=False)
    r2 = r2_score(y_train, y_train_pred)

    # Extract hyperparameters
    lengthscales = model.kern.lengthscale.values
    variance = model.kern.variance.values[0]
    noise = model.likelihood.variance.values[0]

    results[kname] = {
        "Q2": q2,
        "RMSE": rmse,
        "R2": r2,
        "train_time_sec": train_time,
        "lengthscales": lengthscales,
        "variance": variance,
        "noise": noise
    }
    
for k, v in results.items():
    print(f"\nKernel: {k}")
    print("Train time (sec):", round(v["train_time_sec"], 3))
    print("Q2:", v["Q2"])
    print("RMSE:", v["RMSE"])
    print("R2:", v["R2"])
    print("Lengthscales:", v["lengthscales"])
    print("Kernel variance:", v["variance"])
    print("Noise variance:", v["noise"])

Optimization restart 1/5, f = 6.9262106431850015
Optimization restart 2/5, f = 6.926210713253369
Optimization restart 3/5, f = 6.926210617605761


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add


Optimization restart 4/5, f = 7.202087661472149
Optimization restart 5/5, f = 7.488699753411339
Optimization restart 1/5, f = -0.03680984574236845
Optimization restart 2/5, f = -0.03680984573855994
Optimization restart 3/5, f = -0.03680984539400356
Optimization restart 4/5, f = -0.03680984475238347


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:243: RuntimeWarning:invalid value encountered in divide


Optimization restart 5/5, f = 0.8646612885955278
Optimization restart 1/5, f = 3.4163579881648474
Optimization restart 2/5, f = 3.4163579697071214
Optimization restart 3/5, f = 3.416357968740982
Optimization restart 4/5, f = 3.4163579687602095
Optimization restart 5/5, f = 3.4163579687418917
Optimization restart 1/5, f = 9.269288846397586
Optimization restart 2/5, f = 9.269288846544455


 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:166: RuntimeWarning:overflow encountered in divide
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:137: RuntimeWarning:overflow encountered in square
 C:\Users\tateb\anaconda3\envs\bestwick3_gp\lib\site-packages\GPy\kern\src\stationary.py:138: RuntimeWarning:invalid value encountered in add


Optimization restart 3/5, f = 12.753503358822002
Optimization restart 4/5, f = 9.269288846400656
Optimization restart 5/5, f = 9.269288858611247

Kernel: Exponential
Train time (sec): 2.578
Q2: 0.9508618544782043
RMSE: 28.43334789286488
R2: 0.994290583201783
Lengthscales: [164.52853751  34.14930968  61.61745972 295.24539132]
Kernel variance: 8.184041699908782
Noise variance: 0.015132686046575111

Kernel: Matern32
Train time (sec): 2.228
Q2: 0.9518391906871277
RMSE: 28.149164301136597
R2: 0.976451001229794
Lengthscales: [12.20594034  2.94471473  6.25852352 41.29938573]
Kernel variance: 7.00143530013023
Noise variance: 0.030616572738605737

Kernel: Matern52
Train time (sec): 1.858
Q2: 0.9534359272856514
RMSE: 27.67859857701186
R2: 0.9731415139788104
Lengthscales: [ 6.7586598   1.56073114  3.36482391 23.74569657]
Kernel variance: 3.742809014396743
Noise variance: 0.033123003107985646

Kernel: RBF
Train time (sec): 2.141
Q2: 0.9519522952363054
RMSE: 28.116091040418993
R2: 0.966278997246820